# Day03课堂笔记

## 一、测试执行辅助

### 1. HTTP 相关介绍

> **协议**是网络通信使用的一套规则。
>
> **HTTP** 是超文本传输协议，可以传输文字、图片、音频和视频等网络资源，默认端口为 `80`。
>
> **HTTPS** 是安全的超文本传输协议，默认端口为 `443`。

HTTP 通信由请求和响应组成：

- HTTP 请求：请求行、请求头、请求体。
- HTTP 响应：响应行、响应头、响应体。

#### URL

URL 表示网络资源的存储位置，结构为：

```text
协议://域名或IP:端口号/资源路径?查询参数
```

其中 `协议://域名或IP:端口号` 是必需部分；如果端口使用协议默认端口，可以省略端口号。网络协议是计算机通信的规则，常见应用层协议包括 HTTP、SSH 等。

#### HTTP 请求方法与数据类型

| 方法 | 原课中的用途 |
| --- | --- |
| `GET` | 从服务器获取资源（查）。 |
| `POST` | 在服务器新建资源（增）。 |
| `PUT` | 更新服务器资源（改）。 |
| `DELETE` | 删除服务器资源（删）。 |

常见请求体数据类型由请求头中的 `Content-Type` 表示：

- `application/json`：JSON 数据格式。
- `application/x-www-form-urlencoded`：表单默认提交格式。

#### 响应状态码

| 状态码范围 | 含义 | 原课示例 |
| --- | --- | --- |
| `2xx` | 请求成功 | `200` |
| `3xx` | 重定向 | `301`、`302` |
| `4xx` | 客户端错误 | `401`、`403`、`404` |
| `5xx` | 服务器端错误 | `500`、`503` |

#### 面试题：GET 和 POST 的区别

原课从以下角度进行比较：

- **请求体**：POST 通常有请求体，GET 的参数通常放在 URL 查询参数中。
- **参数位置**：POST 参数一般在请求体中，GET 参数在 URL 中。
- **参数大小**：GET 受 URL 长度限制；POST 没有同样的 URL 长度限制，但实际仍受服务器、框架和网关配置限制。
- **可见性**：GET 参数直接出现在 URL 中；POST 参数通常在请求体中。两者都应使用 HTTPS 保护敏感数据，不能把“POST”本身当作安全机制。
- **方法语义**：GET 用于获取资源；POST 常用于提交或创建资源。
- **缓存**：GET 更容易被浏览器或中间缓存；POST 通常不会按 GET 的方式缓存。
- **编码**：GET 参数属于 URL，需遵循 URL 编码；POST 请求体可以按约定使用不同编码格式。

原课还要求准备两个延伸问题：常见 HTTP 响应状态码及含义、HTTP 与 HTTPS 的区别。

### 2. Charles 抓包

**抓包**：使用工具抓取客户端与服务器之间交互的数据包。

主要应用场景：

- **抓包并定位前后端 bug**（重点）。
- 模拟弱网测试。
- 绕过界面，拦截并修改请求，模拟测试后台接口。

Charles 与 Fiddler 的课程对比：Charles 支持 Windows、macOS、Linux 等操作系统；Charles 为收费软件，提供 30 天免费试用。

![Charles 主界面](03-img/image-20220320015732637.png.jpg)

![Charles 会话视图](03-img/image-20211108165741519.png.jpg)

#### 2.1 抓取 HTTPS 数据包的设置

以 Firefox 为例：

1. 在浏览器中设置网络代理。
2. 在 Charles 中打开 `Proxy -- Proxying Settings`，配置代理。

![Firefox 网络设置](03-img/image-20250615225437501.png.jpg)

![Charles Proxying Settings](03-img/image-20250615225204756.png.jpg)

#### 2.2 抓包步骤

```text
抓包步骤：
- 打开 Charles 工具，开启代理设置（Proxy Settings...）。
- 打开默认浏览器（浏览器也需要设置代理），输入待抓包地址并回车或刷新。
- 在会话列表中查看抓取到的数据包，按照域名找到对应数据包。
```

也可以通过 `Proxy --> Proxy setting --> enable transparent HTTP proxying` 开启代理。每次要重新观察某次操作时，可以先用工具栏清除会话信息，再操作浏览器抓取新包。

![开启透明 HTTP 代理](03-img/image-20210917094256869.png.jpg)

课程示例是抓取线上 TPShop 首页数据包并过滤。

#### 2.3 抓包的应用场景

##### 2.3.1 定位前后端 bug

面试题：如何根据抓包工具定位前后端 bug？

案例：TPShop 后台新增会员成功后，详情中的性别与前端页面显示不一致，需要判断应提交给前端还是后端。排查步骤：

1. 在浏览器访问服务器地址，确认 Charles 中能看到数据包。
2. 确定需要抓取的页面和数据来源；本例的数据来自登录成功接口的个人信息响应。
3. 进入登录页面前清空 Charles 中的其他数据包，然后重新登录。
4. 在当前服务器地址下找到登录相关的数据包。
5. 查看请求体，确认请求数据是否与页面输入一致。
6. 查看响应体中与性别有关的字段，并与数据库存储值对比。

定位结论：

- **请求数据有问题 → 前端问题**。
- **响应数据有问题 → 后端问题**。判断响应是否有问题，要比较返回数据与数据库数据是否一致。
- **请求和响应都正确，但页面显示错误 → 前端问题**。

![数据库与抓包响应示例](03-img/image-20250616115909153.png.jpg)

![抓包定位示例](03-img/image-20240907160018283.png.jpg)

##### 2.3.2 弱网设置

通过设置延时和带宽，模拟不同网络环境。

![弱网参数设置](03-img/image-20230308173904327.png.jpg)

使用 Fiddler 时可以按传输 `1KB` 或 `1MB` 数据计算延迟；Charles 也有对应的弱网设置界面。

![Fiddler 延迟计算](03-img/image-20240907165621302.png.jpg)

![Charles 弱网设置](03-img/image-20240907170248765.png.jpg)

具体步骤：

1. 不开启弱网，用正常网络访问页面，抓取数据包并记录传输时间 `Duration`。
2. 配置弱网参数并开启弱网，再访问同一页面，记录 `Duration`。
3. 对比两次时间。弱网下应该出现延迟，但页面不能崩溃。

![弱网下的 Duration 对比](03-img/image-20240907170430501.png.jpg)

##### 2.3.3 绕过界面做服务端测试

当页面限制了异常数据、无法直接输入时，可以绕过界面，通过接口验证服务端逻辑。

例如：

- 注册或登录时测试超过 11 位的手机号；页面可能无法输入，但可以绕过界面测试。
- 购物车商品数量上限为 200；页面无法输入 201 时，可以绕过页面验证接口结果。

两种拦截方向：

- **拦截请求**：测试后台接口的逻辑是否正确。
- **拦截响应**：测试页面显示是否变形或报错。

![拦截请求或响应](03-img/image-20240907172857671.png.jpg)

## 二、软件缺陷

### 不可复现的 bug 怎么办？

原课给出的处理思路是从内部出发向外排查，并结合替换排除法和调试日志：

1. 先按严重级别判断：严重级低时可以暂时记录并继续尝试复现；严重级高时需要分析排查。
2. 回想测试过程：复现步骤是什么，测试环境是否存在差异。
3. 使用**替换法**排除环境、账号、数据或设备差异。
4. 寻求测试老员工或开发协助，记录问题出现时间，查询对应时间段日志。
5. 如果没有日志，请开发提供带调试日志的版本，连续跟踪三个版本；仍未复现时可以暂时放弃。
6. 后续版本再次出现时，转为正式 bug，详细描述复现过程并附上日志信息。

## 三、项目管理工具

### 3.1 禅道介绍

常见项目管理工具包括：

- **禅道**：国产、免费。
- **JIRA**：国外、收费。
- TAPD、QC 等。

禅道演示地址：<https://demo.zentao.net/user-login.html>

禅道的组织特点：

- **三权分立**：产品、开发、测试。
- **四角协同**：产品经理、项目经理、开发团队、测试团队。

### 3.2 测试工作中的应用

- **管理用例**：写用例、评审用例、执行用例。
- **管理缺陷**：写缺陷报告、跟踪缺陷、回归测试。

### 3.3 禅道使用

#### 用例管理

用户入口：`测试 --> 用例 --> 功能用例`。

基本流程：

1. 编写用例。
2. 评审用例。新创建的用例处于待评审状态。
3. 执行用例。用例必须评审通过后才能执行。

![编写禅道用例](03-img/image-20211211164659411-1698996460607-1.png.jpg)

![用例字段示例](03-img/image-20211211164536039-1698996460607-2.png.jpg)

![待评审用例](03-img/image-20211211171912716-1698996460607-3.png.jpg)

![执行已评审用例](03-img/image-20211211171932563-1698996460608-4.png.jpg)

禅道执行结果：

| 结果 | 含义 |
| --- | --- |
| `pass` | 通过 |
| `fail` | 失败 |
| `N/A` | 忽略 |
| `block` | 阻塞 |

![禅道执行结果](03-img/image-20220222185916128-1698996460608-5.png.jpg)

#### 缺陷管理

缺陷流程包括提交 bug、跟踪 bug、验证 bug。

![提交 bug](03-img/image-20211211173242127-1698996460608-6.png.jpg)

![缺陷信息填写](03-img/image-20211211173315891-1698996460608-7.png.jpg)

![跟踪 bug](03-img/image-20220222173817967-1698996460608-8.png.jpg)

![验证 bug](03-img/image-20211211173610023-1698996460608-9.png.jpg)

### 3.4 JIRA 使用（了解）

原课中 JIRA 主要用于 bug 管理：提交 bug、跟踪和验证 bug。

![JIRA 缺陷管理](03-img/image-20240616170356660.png.jpg)

![JIRA 页面示例](03-img/image-20240616170009997.png.jpg)

![JIRA 操作示例](03-img/image-20240616170126203.png.jpg)

#### 扩展示例

![JIRA 扩展](03-img/image-20240616171720160.png.jpg)

## 资料来源

- 课程：`2026最新AI测试基础一套通关`。
- 原始位置：`1、AI测试基础/day03测试执行/01-课堂资料/Day03课堂笔记.md`。
- 配套材料：`Charles抓包工具V2.2-w.pdf`、`缺陷报告模板.xlsx`。
- [百度网盘资料](https://pan.baidu.com/s/11X8sJa4vGgD_jhz4YZ95Ig?pwd=1234)。